In [ ]:
import pandas as pd
import plotnine as gg
import glob
import matplotlib.pyplot as plt
from scipy import stats

from essential.utils import PLOTNINE_DEFAULT_THEME_2

In [ ]:
def _load(pattern):
    return pd.concat(
        [
            pd.read_csv(p).assign(model=p.split("/")[1])
            for p in sorted(glob.glob(pattern))
        ],
        ignore_index=True,
    )


overall_df = _load("results/*/overall_metrics.csv")
perturbation_centric_df = _load("results/*/perturbation_centric_metrics.csv")
gene_centric_df = _load("results/*/gene_centric_metrics.csv")

In [ ]:
gene_centric_df

In [ ]:
plot_df = overall_df.sort_values("lfc_pearson_r", ascending=False)

(
    gg.ggplot(plot_df, gg.aes(x="model", y="lfc_pearson_r"))
    + gg.geom_bar(stat="identity")
    + gg.coord_flip()
)

In [ ]:
plot_df = perturbation_centric_df.loc[
    lambda x: x["model"] == "cellbox_causal_rollout_full_10steps"
]

for xval in ["lfc_norm", "n_cells_gt"]:
    corr = stats.spearmanr(plot_df[xval], plot_df["lfc_pearson_r"])
    fig = (
        gg.ggplot(plot_df, gg.aes(x=xval, y="lfc_pearson_r"))
        + PLOTNINE_DEFAULT_THEME_2
        + gg.geom_point()
    )
    print(
        f"Spearman correlation between {xval} and lfc_pearson_r: {corr.correlation:.2f}, p-value: {corr.pvalue:.2e}"
    )
    display(fig)

In [ ]:
plot_df = gene_centric_df.loc[
    lambda x: x["model"] == "cellbox_causal_rollout_full_10steps"
]

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad")
lib_size = adata.X.sum(axis=1).A1
gene_info = pd.DataFrame(
    {
        "gene": adata.var_names,
        "mean_expression": adata.X.mean(axis=0).A1,
        "n_cells_gt": (adata.X > 0).sum(axis=0).A1,
        "library_size": lib_size.mean(),
    }
)

In [ ]:
plot_df = gene_centric_df.loc[
    lambda x: x["model"] == "cellbox_causal_rollout_full_10steps"
].merge(gene_info, left_on="gene", right_on="gene")
plot_df

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="n_cells_gt", y="r2"))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.geom_point()
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="mean_expression", y="r2"))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.geom_point()
)